# RoadGuard — Full YOLO Pothole Training

This notebook is the **full model-training version**. It is configured with:

```python
RUN_FULL_TRAINING = True
RUN_TRAINING = True
```

Run it on an x86_64 Linux computer or Google Colab. Standard PyTorch wheels are not available for the s390x architecture used by the current LinuxONE environment, so this notebook stops early on s390x instead of producing a misleading partial result.

## Install requirements

On x86_64 Linux, run in a terminal:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
pip install -r requirements-full-training.txt
jupyter lab
```

For Google Colab, run the install cell below and then restart the runtime if Colab requests it.

In [ ]:
import sys, platform, subprocess
from pathlib import Path

ARCHITECTURE = platform.machine().lower()
print("Architecture:", ARCHITECTURE)

if ARCHITECTURE == "s390x":
    raise RuntimeError(
        "This full YOLO notebook requires PyTorch, but standard PyTorch wheels "
        "are not available for s390x. Run this notebook on x86_64 Linux or Google Colab."
    )

try:
    import torch
    import ultralytics
    print("PyTorch:", torch.__version__)
    print("Ultralytics:", ultralytics.__version__)
except ImportError:
    print("Installing full-training packages for the current notebook kernel...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements-full-training.txt"])
    print("Restart the kernel, then run this cell again.")
    raise

In [ ]:
# Google Colab users can run this instead of the terminal install.
# Uncomment the next line only in Colab or another notebook environment that needs it.
# %pip install -r requirements-full-training.txt

## 1. Set the project paths and training flags

In [ ]:
from pathlib import Path
import os, sys, platform, subprocess, zipfile, urllib.request

# These are intentionally True for the full-training notebook.
RUN_FULL_TRAINING = True
RUN_TRAINING = True

# If the automatic download returns HTTP 403, manually place the archive here:
# data/raw/RDD2022_India.zip
LOCAL_ARCHIVE = None


def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/RoadGuard_GitHub_Ready')]
    for candidate in candidates:
        if (candidate / 'scripts' / 'prepare_rdd2022.py').exists() and (candidate / 'data').exists():
            return candidate
    # Colab fallback: create/use a local project folder in the current directory.
    return Path.cwd()

PROJECT = find_project_root()
DATA_ROOT = PROJECT / 'data'
PROCESSED = DATA_ROOT / 'processed' / 'rdd2022_pothole'
RUNS = PROJECT / 'runs'
RUNS.mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT)
print('Data root:', DATA_ROOT)
print('Architecture:', platform.machine())
print('RUN_FULL_TRAINING:', RUN_FULL_TRAINING)
print('RUN_TRAINING:', RUN_TRAINING)

## 2. Prepare the full RDD2022 India dataset

This uses the included XML-to-YOLO converter. It preserves the RDD2022 `D40` pothole class and creates an 80/20 train-validation split. The archive is not committed to GitHub because it may be large and redistribution terms must be reviewed.

In [ ]:
script = PROJECT / 'scripts' / 'prepare_rdd2022.py'
if not script.exists():
    raise FileNotFoundError(f'Missing dataset script: {script}')

archive = DATA_ROOT / 'raw' / 'RDD2022_India.zip'
archive.parent.mkdir(parents=True, exist_ok=True)

if LOCAL_ARCHIVE:
    local = Path(LOCAL_ARCHIVE).expanduser().resolve()
    if not local.exists():
        raise FileNotFoundError(f'LOCAL_ARCHIVE does not exist: {local}')
    archive.write_bytes(local.read_bytes())
    print('Copied local archive to:', archive)

try:
    subprocess.run([
        sys.executable, str(script),
        '--root', str(DATA_ROOT),
        '--val-ratio', '0.2',
        '--seed', '42',
    ], check=True)
except subprocess.CalledProcessError as error:
    print('Dataset preparation failed.')
    print('If the download returned HTTP 403, download RDD2022_India.zip manually and set LOCAL_ARCHIVE above.')
    raise

DATA_YAML = PROCESSED / 'dataset.yaml'
if not DATA_YAML.exists():
    raise FileNotFoundError(f'Expected dataset YAML was not created: {DATA_YAML}')
print(DATA_YAML.read_text())

## 3. Inspect the prepared dataset

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

for split in ['train', 'val']:
    image_dir = PROCESSED / 'images' / split
    label_dir = PROCESSED / 'labels' / split
    images = list(image_dir.glob('*'))
    labels = list(label_dir.glob('*.txt'))
    boxes = sum(len(p.read_text().strip().splitlines()) for p in labels if p.read_text().strip())
    positives = sum(bool(p.read_text().strip()) for p in labels)
    print(f'{split}: {len(images)} images | {positives} positive images | {boxes} pothole boxes')

sample_images = list((PROCESSED / 'images' / 'train').glob('*'))[:12]
if sample_images:
    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    for axis in axes.ravel():
        axis.axis('off')
    for axis, path in zip(axes.ravel(), sample_images):
        axis.imshow(Image.open(path))
        axis.set_title(path.name, fontsize=8)
        axis.axis('off')
    plt.tight_layout()

## 4. Train YOLO

This is the actual model-training step. On CPU it may take a long time. On GPU it is faster. The first run may download `yolo11n.pt`.

In [ ]:
import torch
from ultralytics import YOLO

if not RUN_TRAINING:
    raise RuntimeError('RUN_TRAINING is False. This full-training notebook expects it to be True.')

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
EPOCHS = 30
IMG_SIZE = 640
BATCH = 16 if DEVICE == 0 else 4

print('Device:', 'CUDA GPU' if DEVICE == 0 else 'CPU')
print('Epochs:', EPOCHS)
print('Image size:', IMG_SIZE)
print('Batch:', BATCH)

model = YOLO('yolo11n.pt')
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    project=str(RUNS),
    name='pothole_yolo',
    pretrained=True,
    patience=10,
    seed=42,
    workers=2,
    plots=True,
)

BEST = RUNS / 'pothole_yolo' / 'weights' / 'best.pt'
print('Best model:', BEST)
print('Exists:', BEST.exists())

## 5. Validate and record real metrics

In [ ]:
best_model = YOLO(str(BEST))
metrics = best_model.val(data=str(DATA_YAML), imgsz=IMG_SIZE, device=DEVICE, plots=True)
print(f'mAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print('Precision:', metrics.box.p)
print('Recall:', metrics.box.r)

## 6. Run real model prediction on a validation image

In [ ]:
val_images = list((PROCESSED / 'images' / 'val').glob('*'))
if not val_images:
    raise FileNotFoundError('No validation images found.')

results = best_model.predict(
    source=str(val_images[0]),
    conf=0.25,
    iou=0.5,
    save=True,
    project=str(RUNS),
    name='predict',
    device=DEVICE,
)

annotated = results[0].plot()
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 8))
plt.imshow(annotated[..., ::-1])
plt.axis('off')
plt.title('RoadGuard — real YOLO model prediction')
plt.show()
print('Saved under:', RUNS / 'predict')

## 7. Batch, video, webcam, and export

In [ ]:
# Batch inference on validation images
best_model.predict(
    source=str(PROCESSED / 'images' / 'val'),
    conf=0.25, save=True, save_txt=True,
    project=str(RUNS), name='batch_predictions', device=DEVICE
)
print('Batch output:', RUNS / 'batch_predictions')

# To process a real road video, change SOURCE to a local video path and uncomment:
# SOURCE = '/absolute/path/to/road_video.mp4'
# best_model.predict(source=SOURCE, conf=0.30, save=True, project=str(RUNS), name='video_predictions', device=DEVICE)

# To use a webcam on a computer with a camera:
# best_model.predict(source=0, conf=0.30, show=True, save=True)

# Optional ONNX export:
# exported = best_model.export(format='onnx', imgsz=IMG_SIZE)
# print(exported)

## 8. Interpretation

Only report the metrics printed by this notebook after successful full training. The demo loop in the browser is a presentation asset; the annotated image from Step 6 is the actual model-backed output. Test on representative local road imagery before operational use.